<a href="https://colab.research.google.com/github/edwardsnj/genomic-llms-in-practice/blob/main/Workshop/Exercise_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Workshop: Fine-tuning TinyDNABERT for DNA sequence classification tasks
Model: https://huggingface.co/ChengsenWang/TinyDNABERT-20M-V1

Dataset: https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised

1. Load necessary packages

In [3]:
import torch
from typing import Any, Tuple, Union
import tensorflow as tf
import numpy as np
import pandas as pd
import scipy as sp

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    EvalPrediction,
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import softmax

from peft import get_peft_model, LoraConfig, TaskType

2. Load the dataset

In [4]:
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
print(dataset)

README.md:   0%|          | 0.00/7.14k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

H2AFZ/train.parquet:   0%|          | 0.00/14.6M [00:00<?, ?B/s]

H3K36me3/train.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

...

splice_sites_all/test.parquet:   0%|          | 0.00/903k [00:00<?, ?B/s]

splice_sites_donors/test.parquet:   0%|          | 0.00/894k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/493242 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38822 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 493242
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 38822
    })
})


  a. Separate the train and test datasets.

In [19]:
# Train dataset
train_dataset = dataset["train"]

# Evaluation dataset
test_dataset = dataset["test"]

  b. Filter only for the promoter classification task. This is a binary classification task which involves categorising each sample as "promoter" (1) or "not promoter" (0).

  *Hint: use the `filter` function*

In [21]:
# print(set(train_dataset.to_pandas()["task"]))
train_dataset = train_dataset.filter(lambda example: example["task"] == "promoter_all")
test_dataset = test_dataset.filter(lambda example: example["task"] == "promoter_all")

Filter:   0%|          | 0/30000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1584 [00:00<?, ? examples/s]

b. Split the train dataset further into train (80%) and validation (20%).

*Hint: use the `train_test_split` function.*

In [22]:
# Split the training data into training and validation
split_dataset = train_dataset.train_test_split(test_size=0.2)
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 24000
    })
    test: Dataset({
        features: ['sequence', 'name', 'label', 'task'],
        num_rows: 6000
    })
})


In [23]:
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [24]:
print(f"No. train samples: {len(train_dataset)}\nNo. val. samples: {len(val_dataset)}\nNo. test samples: {len(test_dataset)}")

No. train samples: 24000
No. val. samples: 6000
No. test samples: 1584


d. To make the training less time-consuming, we will use only the first 10000 rows of the training data.

*Hint: use the `select` function.*

In [25]:
train_dataset = train_dataset.shuffle(seed=42).select(range(10000))

3. Load the tokeniser and define the `preprocess_function`, which we will use to batch apply tokenisation to our datasets.

In [26]:
# Set the model name from the Huggingface page
MODEL_NAME = "ChengsenWang/TinyDNABERT-20M-V1"

In [27]:
# Load the tokeniser
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/17.1k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/7.89k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

In [28]:
# Define the preprocessing function
def preprocess_function(examples):
    return tokeniser(
        examples["sequence"],
        truncation=True,
        padding="max_length",
        max_length=512,
        # YOUR CODE HERE: SET THE MAXIMUM LENGTH AS 512
        # YOUR CODE HERE: PAD THE TOKENISED SEQUENCE TO THE MAXIMUM LENGTH
    )

4. Tokenise the data

*Hint: use the `map` function*

In [31]:
train_ds_encoded = train_dataset.map(preprocess_function,batched=True)
val_ds_encoded = val_dataset.map(preprocess_function,batched=True)
test_ds_encoded = test_dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1584 [00:00<?, ? examples/s]

5. Fine-tune for the promoter classification task

In [32]:
# Load the pretrained weights and add a classification head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

model.safetensors:   0%|          | 0.00/79.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: ChengsenWang/TinyDNABERT-20M-V1
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
# Select the training arguments
training_args = TrainingArguments(
    output_dir="tinydnabert-finetuned",
    # YOUR CODE HERE: TRAIN THE MODEL FOR 3 EPOCHS
    num_train_epochs=3,
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 16
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    # YOUR CODE HERE: SET THE LEARNING RATE TO 2E-5
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [41]:
# Set up the trainer with the training arguments above
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=val_ds_encoded
    )

In [43]:
trainer.train()
trainer.save_model("./tinydnabert-finetuned")
tokeniser.save_pretrained("./tinydnabert-finetuned")
trainer.evaluate()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
2.314971,0.319099,3


{'eval_loss': 0.31909918785095215}

#### Evaluate on promoter classification

In [44]:
# TrainingArguments for evaluation only
training_args_eval= TrainingArguments(
    output_dir="./results",
    # YOUR CODE HERE: SET THE BATCH SIZE PER DEVICE TO 24
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    # YOUR CODE HERE: SET UP SO THAT WE DO EVALUATION BUT NOT TRAINING
    do_train=False,
    logging_dir="./logs",
    report_to="none",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [45]:
# Define metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    # Handle output as tuple (logits,) or ModelOutput
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = logits.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    probabilities = softmax(logits, axis=-1)[:, 1] # probability for class 1
    auroc = roc_auc_score(labels, probabilities)  # using sklearn
    return {"accuracy": acc, "auroc": auroc, "f1": f1}

In [46]:
trainer_eval = Trainer(
    model=model,
    args=training_args_eval,
    eval_dataset=test_ds_encoded,
    compute_metrics=compute_metrics,
)

In [68]:
# Evaluate model
eval_results = trainer_eval.evaluate()
print(eval_results)

shuffle = test_dataset.shuffle(seed=42)[:100]
print(shuffle["label"])
preds = []
for seq in shuffle["sequence"]:
  enc = tokeniser(
        seq,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
  ).to("cuda")
  with torch.no_grad():
    out = model(**enc)
  preds.append(out.logits.to("cpu")[0].argmax().item())
print(preds)


[1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0]
[1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1]
